# Tutorial: Astra DB StreamFlix Workshop

This notebook seeds a Netflix-style demo with:
- `shows` collection (vectorized docs)
- `user_session_events` table
- `home_rails_by_profile` table

By the end, your frontend and backend will have realistic data for semantic search and personalization.


## Audience, Prerequisites, and Learning Goals

**Audience**: Developers running the 1-hour StreamFlix workshop.

**Prerequisites**:
- Astra DB Serverless database provisioned (AWS `us-east-2` recommended for Astra-hosted NVIDIA vectorize)
- Application token + Data API endpoint copied from Astra Portal
- `data/tv_shows_300.json` and `data/seed_profiles.json` available in this repo
- No TMDB key required for participants (snapshot is already pre-enriched offline)

**Learning goals**:
1. Connect to Astra Data API from Python.
2. Create vector collection and two query-optimized tables.
3. Load realistic TV metadata, rails, and session events.
4. Validate semantic search and home-rail query patterns.


## Outline

1. Install/import dependencies and read environment variables.
2. Connect to Astra DB and create collection + tables.
3. Load local snapshot data and profile fixtures.
4. Insert documents/rows in batches.
5. Validate counts and run a semantic query.
6. Complete one exercise + one optional extension.


In [1]:
# If needed, uncomment and run once:
# %pip install -q astrapy python-dotenv

import json
import os
from pathlib import Path

from astrapy import DataAPIClient
from astrapy.constants import SortMode
from astrapy.info import CollectionDefinition, CreateTableDefinition, ColumnType


In [2]:
ASTRA_DB_API_ENDPOINT = os.getenv('ASTRA_DB_API_ENDPOINT', '').strip()
ASTRA_DB_APPLICATION_TOKEN = os.getenv('ASTRA_DB_APPLICATION_TOKEN', '').strip()
ASTRA_DB_KEYSPACE = os.getenv('ASTRA_DB_KEYSPACE', 'default_keyspace').strip()

if not ASTRA_DB_API_ENDPOINT or not ASTRA_DB_APPLICATION_TOKEN:
    raise ValueError('Set ASTRA_DB_API_ENDPOINT and ASTRA_DB_APPLICATION_TOKEN in your environment before continuing.')

client = DataAPIClient(ASTRA_DB_APPLICATION_TOKEN)
db = client.get_database(ASTRA_DB_API_ENDPOINT, keyspace=ASTRA_DB_KEYSPACE)
print(f'Connected to Astra endpoint: {ASTRA_DB_API_ENDPOINT}')
print(f'Using keyspace: {ASTRA_DB_KEYSPACE}')


Connected to Astra endpoint: https://59b9a1f3-7330-49ea-af36-d3f8c75da9c5-us-east-2.apps.astra.datastax.com
Using keyspace: default_keyspace


In [3]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'tv_shows_300.json').exists():
            return candidate
    raise FileNotFoundError('Could not find data/tv_shows_300.json from notebook location')

repo_root = find_repo_root(Path.cwd())
shows_path = repo_root / 'data' / 'tv_shows_300.json'
seeds_path = repo_root / 'data' / 'seed_profiles.json'

shows_seed = json.loads(shows_path.read_text(encoding='utf-8'))
profiles_seed = json.loads(seeds_path.read_text(encoding='utf-8'))
print(f'Loaded {len(shows_seed)} show documents and {len(profiles_seed["profiles"])} profile fixtures')


Loaded 300 show documents and 3 profile fixtures


In [4]:
collection_name = 'shows'
existing_collections = set(db.list_collection_names())
print('Collections before create:', sorted(existing_collections))

if collection_name not in existing_collections:
    definition = (
        CollectionDefinition.builder()
        .set_vector_service('nvidia', 'NV-Embed-QA')
        .build()
    )
    db.create_collection(collection_name, definition=definition)
    print('Created vectorized collection: shows')
else:
    print('Collection already exists: shows')

existing_collections = set(db.list_collection_names())
print('Collections after create:', sorted(existing_collections))

shows_collection = db.get_collection(collection_name)
print('Using collection:', collection_name)



Collections before create: ['cimb_malaysia', 'costco_8k_march2nd', 'costco_sec8k_collection', 'ctbc_demo_dec05', 'sec8k', 'sec_ingestion_registry', 'shows']
Collection already exists: shows
Collections after create: ['cimb_malaysia', 'costco_8k_march2nd', 'costco_sec8k_collection', 'ctbc_demo_dec05', 'sec8k', 'sec_ingestion_registry', 'shows']
Using collection: shows


In [5]:
sessions_definition = (
    CreateTableDefinition.builder()
    .add_column('profile_id', ColumnType.TEXT)
    .add_column('event_day', ColumnType.TEXT)
    .add_column('event_ts', ColumnType.BIGINT)
    .add_column('event_id', ColumnType.TEXT)
    .add_column('show_id', ColumnType.TEXT)
    .add_column('event_type', ColumnType.TEXT)
    .add_column('progress_seconds', ColumnType.INT)
    .add_column('device_type', ColumnType.TEXT)
    .add_column('locale', ColumnType.TEXT)
    .add_partition_by(['profile_id', 'event_day'])
    .add_partition_sort({'event_ts': SortMode.DESCENDING, 'event_id': SortMode.ASCENDING})
    .build()
)

rails_definition = (
    CreateTableDefinition.builder()
    .add_column('profile_id', ColumnType.TEXT)
    .add_column('rail_id', ColumnType.TEXT)
    .add_column('rank', ColumnType.INT)
    .add_column('show_id', ColumnType.TEXT)
    .add_column('reason', ColumnType.TEXT)
    .add_partition_by(['profile_id', 'rail_id'])
    .add_partition_sort({'rank': SortMode.ASCENDING})
    .build()
)

db.create_table('user_session_events', definition=sessions_definition, if_not_exists=True)
db.create_table('home_rails_by_profile', definition=rails_definition, if_not_exists=True)

session_table = db.get_table('user_session_events')
rails_table = db.get_table('home_rails_by_profile')
print('Tables ready: user_session_events, home_rails_by_profile')


Tables ready: user_session_events, home_rails_by_profile


In [6]:
show_documents = []
for row in shows_seed:
    show_documents.append(
        {
            '_id': row['_id'],
            'title': row['title'],
            'genres': row.get('genres', []),
            'year': row.get('year'),
            'premiered_date': row.get('premiered_date'),
            'status': row.get('status'),
            'synopsis': row.get('synopsis', ''),
            'poster_url': row.get('poster_url', ''),
            'rating': row.get('rating'),
            'network': row.get('network'),
            'language': row.get('language', 'English'),
            'runtime': row.get('runtime', 0),
            'row_tags': row.get('row_tags', []),
            'tags': row.get('tags', []),
            'creator_names': row.get('creator_names', []),
            'director_names': row.get('director_names', []),
            'cast_names': row.get('cast_names', []),
            '$vectorize': row.get('vectorize_text', ''),
        }
    )

# Re-seed for deterministic workshop runs.
shows_collection.delete_many({})
shows_collection.insert_many(show_documents, ordered=False, chunk_size=20)

print(f'Inserted {len(show_documents)} show docs into `shows` collection')



Inserted 300 show docs into `shows` collection


In [7]:
rails_rows = profiles_seed['rails_rows']
session_rows = profiles_seed['session_rows']

# Re-seed for deterministic workshop runs.
# Use partition-key filters (no full-table scans).
rails_partitions = {
    (row['profile_id'], row['rail_id'])
    for row in rails_rows
}
for profile_id, rail_id in sorted(rails_partitions):
    rails_table.delete_many({'profile_id': profile_id, 'rail_id': rail_id})

session_partitions = {
    (row['profile_id'], row['event_day'])
    for row in session_rows
}
for profile_id, event_day in sorted(session_partitions):
    session_table.delete_many({'profile_id': profile_id, 'event_day': event_day})

rails_table.insert_many(rails_rows, ordered=False, chunk_size=100)
session_table.insert_many(session_rows, ordered=False, chunk_size=100)

print(f'Inserted {len(rails_rows)} rows into home_rails_by_profile')
print(f'Inserted {len(session_rows)} rows into user_session_events')



Inserted 180 rows into home_rails_by_profile
Inserted 24 rows into user_session_events


In [8]:
show_count = shows_collection.count_documents({}, upper_bound=10_000)

# For tables, avoid zero-filter scans. Validate with partition-key reads.
expected_rails_count = len(rails_rows)
expected_session_count = len(session_rows)

sample_profile = profiles_seed['profiles'][0]['profile_id']
sample_rail = 'trending'
sample_day = session_rows[0]['event_day'] if session_rows else None

sample_rail_rows = list(
    rails_table.find(
        {'profile_id': sample_profile, 'rail_id': sample_rail},
        limit=20,
    )
)

sample_session_rows = []
if sample_day:
    sample_session_rows = list(
        session_table.find(
            {'profile_id': sample_profile, 'event_day': sample_day},
            limit=20,
        )
    )

print('Counts (from inserted payload):')
print(f'  shows: {show_count}')
print(f'  home_rails_by_profile: {expected_rails_count}')
print(f'  user_session_events: {expected_session_count}')

print('\nPartition validation:')
print(f'  rails partition sample ({sample_profile}, {sample_rail}) -> {len(sample_rail_rows)} rows')
if sample_day:
    print(f'  session partition sample ({sample_profile}, {sample_day}) -> {len(sample_session_rows)} rows')

def present(value):
    if value is None:
        return False
    if isinstance(value, str):
        return bool(value.strip())
    if isinstance(value, list):
        return bool(value)
    if isinstance(value, (int, float)):
        return value > 0
    return True

metadata_fields = [
    'network',
    'runtime',
    'language',
    'status',
    'premiered_date',
    'creator_names',
    'director_names',
    'cast_names',
]

print('\nMetadata fill-rate snapshot:')
for field in metadata_fields:
    filled = sum(1 for row in show_documents if present(row.get(field)))
    pct = (filled / len(show_documents)) * 100 if show_documents else 0
    print(f'  {field:16} {filled:3d}/{len(show_documents)} ({pct:5.1f}%)')

semantic_query = 'dark political prison drama with strong characters'
semantic_results = list(
    shows_collection.find(
        {},
        sort={'$vectorize': semantic_query},
        include_similarity=True,
        limit=5,
    )
)

print('\nSemantic query sample results:')
for idx, doc in enumerate(semantic_results, start=1):
    print(f"{idx}. {doc.get('title')} | similarity={doc.get('$similarity')}")

if semantic_results:
    sample_doc = semantic_results[0]
    print('\nSample metadata payload:')
    print('  title:', sample_doc.get('title'))
    print('  creators:', sample_doc.get('creator_names', []))
    print('  directors:', sample_doc.get('director_names', []))
    print('  cast:', sample_doc.get('cast_names', [])[:5])
    print('  tags:', sample_doc.get('tags', []))


Counts (from inserted payload):
  shows: 300
  home_rails_by_profile: 180
  user_session_events: 24

Partition validation:
  rails partition sample (profile_alex, trending) -> 12 rows
  session partition sample (profile_alex, 2026-03-17) -> 4 rows

Semantic query sample results:
1. Alcatraz | similarity=0.7413187
2. State of Affairs | similarity=0.7338771
3. Believe | similarity=0.73298603
4. The Bridge | similarity=0.72250813
5. Rectify | similarity=0.72182786

Sample metadata payload:
  title: Alcatraz
  creators: ['Steven Lilien', 'Bryan Wynbrandt', 'Elizabeth Sarnoff']
  directors: []
  cast: ['Sarah Jones', 'Jorge Garcia', 'Sam Neill', 'Jonathan Coyne', 'Parminder Nagra']
  tags: ['binge', 'drama', 'sci_fi', 'scripted', 'english', 'ended']


## Exercise

Change `semantic_query` to one of the prompts below and re-run the validation cell:
- "feel-good family comedy with music"
- "space war political thriller"
- "medical drama in a big city hospital"

Then confirm the top results look semantically relevant.


## Pitfall and Extension

**Common pitfall**: wrong keyspace or missing environment variable causes table/collection creation to fail.

**Extension**: Add a new rail (`crime`) by tagging relevant shows in `tv_shows_300.json`, regenerating `seed_profiles.json`, and re-running this notebook.


## Attribution

TV metadata and image URLs are sourced from [TVMaze API](https://www.tvmaze.com/api) for workshop/demo use. Check TVMaze terms before publishing outside the workshop context.

Offline metadata enrichment for the local snapshot may additionally use [TMDB API](https://developer.themoviedb.org/) during maintenance refresh runs.
